# S03 — Git, GitHub & Project Structure

**Python for Applied Physics** · Master in Applied Physics  
Session 3 of 10 · 2 hours

---

## Session roadmap

| # | Topic | Time |
|---|-------|------|
| 1 | Why version control? The mental model | 10 min |
| 2 | Local Git: init, stage, commit, log, diff | 25 min |
| 3 | Branching: create, switch, merge | 15 min |
| 4 | GitHub: remote, push, pull, clone, fork, PR | 20 min |
| 5 | Scientific Python project structure | 10 min |
| — | Exercises | 40 min |

---

**Note on this notebook:** Git is a terminal tool. This notebook interleaves explanations (Markdown cells) with terminal commands that you run in VS Code's integrated terminal — not as Python code cells. Commands are shown in `bash` code blocks. Python cells are used only where Python genuinely helps (e.g., generating a `.gitignore`, inspecting a repo programmatically).

**Open your terminal now:** in VS Code, press `` Ctrl+` `` (Windows/Linux) or `` Ctrl+` `` (macOS) to open the integrated terminal.

---
## 1 · Why version control?

You have almost certainly already invented a primitive version control system:

```
simulation_v1.py
simulation_v2.py
simulation_v2_final.py
simulation_v2_final_REAL.py
simulation_v2_final_REAL_fixed.py
```

This breaks down immediately when:
- You need to know *what* changed between versions
- You need to go back to a state from three weeks ago
- Two people edit the same file
- You want to try a risky change without breaking the working version

Git solves all of these.

### The mental model: snapshots, not diffs

Git stores the **complete state of your project** at each commit — a snapshot. Each snapshot has:
- A unique identifier (SHA hash, e.g. `a3f9c2b`)
- A pointer to its parent snapshot
- Your commit message
- The author and timestamp

This chain of snapshots is your project's history. You can jump to any snapshot at any time.

```
  [initial setup]  →  [add Fresnel functions]  →  [fix Brewster angle bug]  →  [add ABCD propagation]
       a1b2c3              d4e5f6                        7g8h9i                       j0k1l2
```

### Three areas

Understanding Git's three areas prevents 90% of confusion:

| Area | What it is |
|------|------------|
| **Working directory** | Your actual files on disk — what you see and edit |
| **Staging area (index)** | Files you've selected for the next commit (`git add`) |
| **Repository (`.git/`)** | The history of all commits |

---
## 2 · Local Git

### 2.1 · One-time global setup

Run these once after installing Git. This sets your identity on every commit.

```bash
git config --global user.name  "Your Name"
git config --global user.email "you@example.com"
git config --global core.editor "code --wait"   # VS Code as the commit message editor
git config --global init.defaultBranch main
```

Verify your settings:
```bash
git config --list
```

### 2.2 · Creating a repository

Navigate to the folder you want to version-control and run:

```bash
cd ~/your-project-folder
git init
```

This creates a hidden `.git/` folder — the repository database. Your files are untouched.

Check the current status at any time:
```bash
git status
```

You will see all your files listed as *untracked* — Git knows they exist but isn't tracking them yet.

### 2.3 · `.gitignore` — what NOT to track

Before your first commit, create a `.gitignore` file in the project root. This tells Git to completely ignore certain files. You never want to commit:
- Python bytecode (`__pycache__/`, `*.pyc`)
- Virtual environment folders (`venv/`, `.conda/`)
- Jupyter checkpoint folders (`.ipynb_checkpoints/`)
- Large data files (`.npy`, `.h5`, `.mat`) — data goes in a separate storage or is regenerated
- IDE/OS files (`.vscode/settings.json`, `.DS_Store`)

The cell below generates a sensible `.gitignore` for a scientific Python project:

In [2]:
gitignore_content = """# Python bytecode
__pycache__/
*.py[cod]
*$py.class
*.so

# Virtual environments
.venv/
venv/
env/
.conda/

# Jupyter
.ipynb_checkpoints/
*.ipynb_checkpoints

# Large data files (store separately, regenerate from code)
*.npy
*.npz
*.h5
*.hdf5
*.mat
*.csv
data/raw/

# Build / packaging
dist/
build/
*.egg-info/

# IDE & OS
.vscode/
.idea/
.DS_Store
Thumbs.db

# Secrets — NEVER commit credentials
.env
secrets.py
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

print(".gitignore written. Contents:")
print(gitignore_content)

.gitignore written. Contents:
# Python bytecode
__pycache__/
*.py[cod]
*$py.class
*.so

# Virtual environments
.venv/
venv/
env/
.conda/

# Jupyter
.ipynb_checkpoints/
*.ipynb_checkpoints

# Large data files (store separately, regenerate from code)
*.npy
*.npz
*.h5
*.hdf5
*.mat
*.csv
data/raw/

# Build / packaging
dist/
build/
*.egg-info/

# IDE & OS
.vscode/
.idea/
.DS_Store
Thumbs.db

# Secrets — NEVER commit credentials
.env
secrets.py



> **⚠️ Common Pitfall — Committing large data files**  
> Git stores every version of every file forever. A 50 MB `.npy` array accidentally committed will bloat the repository permanently (even if you later delete the file). Always add data formats to `.gitignore` before your first commit. If you need to share large datasets, use cloud storage (Zenodo, OSF, institutional storage) and add a `data/README.md` explaining where to download them.

### 2.4 · Stage and commit

The workflow for every change:

```bash
# See what has changed
git status

# Stage specific files
git add fresnel.py
git add notebooks/S02_lecture.ipynb

# Or stage everything in the current folder
git add .

# Commit with a message
git commit -m "Add Fresnel reflectance function with Brewster angle finder"
```

**Writing good commit messages:**
- Use the imperative mood: *"Add"*, *"Fix"*, *"Remove"*, not *"Added"* or *"Adding"*
- Complete the sentence: *"If applied, this commit will..."*
- Keep the first line under 72 characters
- Reference the physics when relevant: *"Fix off-by-one in Rayleigh range calculation"*

| ❌ Bad | ✅ Good |
|--------|--------|
| `update` | `Fix unit conversion bug in spectral_table (nm → m)` |
| `wip` | `Add ABCD matrix propagation for Gaussian beams` |
| `asdf` | `Remove debug print statements from Fresnel module` |

### 2.5 · Inspecting history

```bash
# Full log
git log

# Compact one-line-per-commit view
git log --oneline

# With a visual branch graph
git log --oneline --graph --all

# Show what changed in the last commit
git show HEAD

# Show what changed between two commits
git diff a3f9c2b d4e5f6a

# Show what you've changed but not yet staged
git diff

# Show what's staged but not yet committed
git diff --staged
```

> **💡 Pro Tip — GitLens in VS Code**  
> The GitLens extension (already in your extension stack) shows Git blame inline — who changed each line and when — without leaving the editor. Open the Source Control panel (`Ctrl+Shift+G`) to see your commit history visually, stage files by clicking `+`, and write commit messages in the text box at the top.

### 2.6 · Undoing things

This is where people get anxious. Git makes it very hard to permanently lose work.

```bash
# Unstage a file (undo git add, keep your changes)
git restore --staged fresnel.py

# Discard changes in working directory (IRREVERSIBLE — file goes back to last commit)
git restore fresnel.py

# Amend the last commit message (before pushing)
git commit --amend -m "Better commit message"

# Create a new commit that reverses a previous commit (safe — preserves history)
git revert a3f9c2b

# Go back to a previous state to LOOK at it (detached HEAD — read-only exploration)
git checkout a3f9c2b
git checkout main          # return to the present
```

> **⚠️ Common Pitfall — `git reset --hard`**  
> `git reset --hard <commit>` moves the branch pointer AND discards all uncommitted work. It is the one Git command that can cause real data loss. Avoid it until you understand it well. Use `git revert` instead — it achieves the same logical result safely by adding a new commit.

---
## 3 · Branching

A branch is a lightweight, movable pointer to a commit. Creating one is instantaneous and costs almost nothing. Branches let you:
- Work on a new feature without breaking the working code
- Try a risky numerical experiment safely
- Maintain a clean `main` branch that always runs correctly

### 3.1 · Create and switch

```bash
# List all branches (* marks current)
git branch

# Create a new branch
git branch feature/b-integral

# Switch to it
git switch feature/b-integral

# Create AND switch in one command (preferred)
git switch -c feature/b-integral
```

Now any commits you make are on `feature/b-integral`. The `main` branch is untouched.

### 3.2 · Merge

When your feature is ready, merge it back into `main`:

```bash
# Switch back to main
git switch main

# Merge the feature branch into main
git merge feature/b-integral

# Clean up: delete the branch (the commits are now in main)
git branch -d feature/b-integral
```

### 3.3 · Resolving a merge conflict

A conflict occurs when the same lines were changed in both branches. Git marks the conflict in the file:

```python
<<<<<<< HEAD
    return w0 * math.sqrt(1 + (z / z_R)**2)   # your version in main
=======
    return w0 * (1 + (z / z_R)**2)**0.5       # version from feature branch
>>>>>>> feature/b-integral
```

To resolve:
1. Open the file in VS Code — it shows a visual merge editor with **Accept Current**, **Accept Incoming**, or **Accept Both** buttons
2. Choose the correct version (or combine them manually)
3. Save, then `git add` the file and `git commit`

> **🔬 Physics Insight — Branches for computational experiments**  
> In simulation code, branches are ideal for exploring numerical methods. Keep `main` running your validated split-step NLSE solver. Create `experiment/adaptive-step` to try a new algorithm. If it improves accuracy, merge. If not, delete the branch. Your validated solver is always one `git switch main` away.

---
## 4 · GitHub

GitHub is a hosting service for Git repositories. It adds:
- Remote backup of your code
- Collaboration (pull requests, code review, issues)
- Visibility (share your research code with the community)
- CI/CD integration (automated testing — covered in S10)

### 4.1 · Authentication: SSH key setup

SSH keys let you push and pull without typing your password every time.

```bash
# Generate a key pair (accept defaults, optionally add a passphrase)
ssh-keygen -t ed25519 -C "you@example.com"

# Copy the public key to your clipboard
# macOS:
cat ~/.ssh/id_ed25519.pub | pbcopy
# Linux:
cat ~/.ssh/id_ed25519.pub   # then copy the output manually
# Windows (PowerShell):
Get-Content ~/.ssh/id_ed25519.pub | clip
```

Then: GitHub → Settings → SSH and GPG keys → New SSH key → paste and save.

Test the connection:
```bash
ssh -T git@github.com
# Should print: Hi username! You've successfully authenticated.
```

### 4.2 · Push an existing local repo to GitHub

1. Create a new **empty** repository on GitHub (no README, no `.gitignore` — you already have these locally)
2. Copy the SSH URL: `git@github.com:yourusername/your-repo.git`

```bash
# Add the remote ("origin" is the conventional name for the main remote)
git remote add origin git@github.com:yourusername/your-repo.git

# Push the main branch and set it as the tracking branch
git push -u origin main
```

After this initial push, future pushes are just:
```bash
git push
```

### 4.3 · Clone an existing repository

To get a copy of any repository (yours or someone else's):

```bash
git clone git@github.com:username/repo-name.git
# or, for public repos without SSH:
git clone https://github.com/username/repo-name.git
```

This creates a folder `repo-name/` with the full history.

### 4.4 · The pull → work → push cycle

When collaborating (or working across machines):

```bash
# 1. Fetch and integrate remote changes before starting work
git pull

# 2. Do your work, stage, commit as usual
git add .
git commit -m "Add transform-limited pulse duration function"

# 3. Push your commits
git push
```

> **💡 Pro Tip — Always pull before you push**  
> If someone else pushed while you were working, `git push` will be rejected. Running `git pull` first fetches their changes and merges them with yours. If there are no conflicts, this is automatic. Get into the habit of pulling at the start of every work session.

### 4.5 · Forks and Pull Requests

**Fork:** A copy of someone else's repository under your GitHub account. Used when you want to contribute to a project you don't have write access to.

**Pull Request (PR):** A proposal to merge your branch (or fork) into another branch. On GitHub this is the standard code review workflow.

Typical open-source contribution workflow:
1. Fork the repository on GitHub
2. `git clone` your fork
3. Create a branch: `git switch -c fix/rayleigh-range-units`
4. Make changes, commit
5. Push to your fork: `git push -u origin fix/rayleigh-range-units`
6. On GitHub: click **Compare & pull request** → write a description → submit
7. The maintainer reviews, requests changes, or merges

For course work, PRs are how you will submit assignments in collaborative exercises.

### 4.6 · Essential GitHub features

| Feature | What it does | Where to find it |
|---------|-------------|------------------|
| **Issues** | Track bugs, feature requests, TODOs | Repository → Issues tab |
| **README.md** | Landing page — what the repo does, how to install/run | Root of repo, auto-rendered |
| **Releases / Tags** | Mark stable versions: `v1.0`, `v2.3.1` | Repository → Releases |
| **GitHub Actions** | Automated testing on every push (covered in S10) | Repository → Actions tab |
| **GitHub Pages** | Host a static website from your repo (documentation) | Settings → Pages |

> **🔬 Physics Insight — Code as a research output**  
> Many journals now require or encourage code availability. A GitHub repository with a clear README, a tag for the paper version, and a Zenodo integration (which gives you a DOI) makes your simulation code a citable research output. Start the habit now: every project gets a repo.

---
## 5 · Scientific Python project structure

From S03 onwards, all course code lives in a structured project. Here is the canonical layout we will use:

```
applied-physics-python/         ← root of your Git repository
│
├── README.md                   ← what this repo is, how to set it up
├── .gitignore                  ← already written above
├── requirements.txt            ← pip install -r requirements.txt
│
├── shared/                     ← reusable modules, imported by notebooks
│   ├── __init__.py
│   ├── optics.py               ← Gaussian beam, ABCD, Fresnel functions
│   └── pulses.py               ← pulse parameter calculations
│
├── S01_ecosystem/
│   └── notebooks/
│       ├── S01_lecture.ipynb
│       ├── S01_exercises.ipynb
│       └── S01_solutions.ipynb
│
├── S02_control_flow/
│   └── notebooks/
│       ├── S02_lecture.ipynb
│       ├── S02_exercises.ipynb
│       └── S02_solutions.ipynb
│
├── S03_git_github/
│   └── notebooks/
│       └── S03_lecture.ipynb   ← this file
│
└── ...                         ← S04 – S10 added progressively
```

### Why the `shared/` folder?

Functions you write in S02 (Fresnel, Gaussian beam, pulse parameters) will be needed again in S04 (NumPy), S05 (Matplotlib), S07 (OOP), and beyond. Rather than copy-pasting code between notebooks, we place reusable functions in `shared/` and import them:

```python
import sys
sys.path.insert(0, "../../shared")   # from any session notebook
from optics import gaussian_beam_waist, rayleigh_range
```

Starting from S03, every function you're happy with goes into `shared/`.

### 5.1 · Creating the `shared/` module

Let's create the initial `shared/optics.py` now with the functions from S02:

In [ ]:
import os

optics_module = '''"""shared/optics.py

Reusable optics functions for the Applied Physics Python course.
Grow this module as the course progresses.
"""

import math


def rayleigh_range(w0, wavelength):
    """
    Rayleigh range of a Gaussian beam.

    Parameters
    ----------
    w0 : float
        Beam waist radius [m].
    wavelength : float
        Laser wavelength [m].

    Returns
    -------
    float
        Rayleigh range z_R [m].
    """
    return math.pi * w0**2 / wavelength


def gaussian_beam_waist(z, w0, wavelength):
    """
    Beam radius of a Gaussian beam at propagation distance z.

    Parameters
    ----------
    z : float
        Propagation distance from the waist [m].
    w0 : float
        Beam waist radius [m].
    wavelength : float
        Laser wavelength [m].

    Returns
    -------
    float
        Beam radius w(z) [m].
    """
    z_R = rayleigh_range(w0, wavelength)
    return w0 * math.sqrt(1 + (z / z_R)**2)


def fresnel_reflectance(n1, n2, theta_i_deg):
    """
    Fresnel reflectance for s- and p-polarizations.

    Parameters
    ----------
    n1, n2 : float
        Refractive indices of incident and transmitted media.
    theta_i_deg : float
        Angle of incidence [degrees].

    Returns
    -------
    R_s, R_p : float
        Reflectance for s- and p-polarizations (0–1).
    """
    theta_i = math.radians(theta_i_deg)
    sin_t = n1 * math.sin(theta_i) / n2
    if abs(sin_t) > 1.0:
        return 1.0, 1.0
    theta_t = math.asin(sin_t)
    cos_i = math.cos(theta_i)
    cos_t = math.cos(theta_t)
    r_s = (n1 * cos_i - n2 * cos_t) / (n1 * cos_i + n2 * cos_t)
    r_p = (n2 * cos_i - n1 * cos_t) / (n2 * cos_i + n1 * cos_t)
    return r_s**2, r_p**2
'''

# Write to ../../shared/ relative to this notebook
# Adjust path if your project structure differs
shared_dir = os.path.join("..", "..", "shared")
os.makedirs(shared_dir, exist_ok=True)

with open(os.path.join(shared_dir, "optics.py"), "w") as f:
    f.write(optics_module)

# Create an __init__.py so shared/ is a proper package
with open(os.path.join(shared_dir, "__init__.py"), "w") as f:
    f.write("")

print("shared/optics.py created.")

In [ ]:
# Verify: import from shared/ and call a function
import sys
import os
sys.path.insert(0, os.path.join("..", "..", "shared"))

from optics import gaussian_beam_waist, rayleigh_range

w0  = 50e-6
lam = 1030e-9
zR  = rayleigh_range(w0, lam)
print(f"z_R = {zR*1e3:.2f} mm")
print(f"w(z_R) = {gaussian_beam_waist(zR, w0, lam)*1e6:.1f} µm  (should be w0·√2 = {w0*1.414e6:.1f} µm)")

### 5.2 · `requirements.txt`

A `requirements.txt` at the project root documents which packages are needed. Anyone cloning your repo can reproduce your environment with `pip install -r requirements.txt`.

In [ ]:
requirements = """# Python for Applied Physics — course dependencies
# Install with: pip install -r requirements.txt

numpy>=1.24
scipy>=1.10
matplotlib>=3.7
pandas>=2.0
h5py>=3.8
jupyter>=1.0
ipykernel>=6.0
"""

with open(os.path.join("..", "..", "requirements.txt"), "w") as f:
    f.write(requirements)

print("requirements.txt written.")
print(requirements)

> **💡 Pro Tip — Pin versions for reproducibility**  
> For a course, `numpy>=1.24` (minimum version) is fine. For a publication, pin exact versions with `numpy==1.26.4` so the paper's results can be reproduced exactly years later. Generate a pinned file with `pip freeze > requirements_frozen.txt`.

---
## 6 · Session summary

| Concept | Key command / action |
|---------|---------------------|
| Initialize a repo | `git init` |
| Check status | `git status` |
| Stage files | `git add <file>` or `git add .` |
| Commit | `git commit -m "message"` |
| View history | `git log --oneline --graph --all` |
| See changes | `git diff`, `git diff --staged` |
| Undo staging | `git restore --staged <file>` |
| Safe undo | `git revert <hash>` |
| Create branch | `git switch -c feature/name` |
| List branches | `git branch`|
| Merge branch | `git switch main` → `git merge feature/name` |
| Connect to GitHub | `git remote add origin <url>` |
| Push | `git push -u origin main` (first time), `git push` (after) |
| Pull | `git pull` |
| Clone | `git clone <url>` |

**From S03 onwards:** every piece of code you write lives in the `applied-physics-python` repository. Commit at the end of every work session, and push to GitHub as your off-site backup.

**Next session (S04):** NumPy — thinking in arrays. The functions in `shared/optics.py` will be rewritten to handle arrays natively.